In [1]:
#importlibraries

#libraries to get files and move directories
import glob
import shutil
import os


#libraries to extract frame
import pandas as pd
import cv2
import numpy as np

#libraries to undergo differencing
import skimage.metrics
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from torchvision.utils import save_image

#create a function that transforms images to tensors
convert_tensor = transforms.ToTensor()
convert_image = transforms.ToPILImage()

c:\Users\MAS203\Anaconda3\envs\diff_cond_env\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#image working directory
img_dir = "../../data/Kakadu_fish/training_data/images/"
#video directory
vid_dir = "../../data/Kakadu_fish/surrounding_videos/"
#where to save scores
score_dir = "../../data/Kakadu_fish/other/matching_images/scores/"
#where location data lies
loc_dir = "../../data/Kakadu_fish/other/matching_images/"
#matching stage 1
match_stage_1 = "../../data/Kakadu_fish/other/matching_images/matching_stage_1.csv"


In [3]:
#load image names and vid locations
img_dat = pd.read_csv(loc_dir + "image_names.csv")
video_dat = pd.read_csv(loc_dir + "vid_locations.csv")
m_st_1_dat = pd.read_csv(match_stage_1)

img_list = img_dat["image_name"]
videos = video_dat["video_location"]


In [7]:
#choose vid and image number
#num_vals = 10
num_vals = len(videos)
#vid_num = 0
#im_num = int(sys.argv[1]) -1
#im_num = 1
#choose frame number/s
#frame_num = 18000
frame_step = 15000

im_num = 0


#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_val = 1000
frame_nums = []



'../../data/Kakadu_fish/surrounding_videos/2017/Sandy Billabong 2017/Transect 6/Location 3/GP020787.MP4'

In [12]:
#Load image
img = Image.open(img_dir +m_st_1_dat["image_name"][im_num])

if(m_st_1_dat["flipped"][im_num]=="yes"):
    img = img.transpose(Image.FLIP_TOP_BOTTOM)

img_tens = convert_tensor(img)


In [43]:
vid_seq = m_st_1_dat["video_location"][im_num].split("/")
vid_seq.pop()
'/'.join(vid_seq) + "/"

'../../data/Kakadu_fish/surrounding_videos/2017/Sandy Billabong 2017/Transect 6/Location 3/'

In [ ]:
video = m_st_1_dat["video_location"][im_num]
vidcap = cv2.VideoCapture(video)
temp_frame = 0

In [32]:
video = m_st_1_dat["video_location"][im_num]
vid = cv2.VideoCapture(video)
num_frames = vidcap.get(cv2.CAP_PROP_FRAME_COUNT)
temp_frame = 0

while(1):
    
      
    # Capture the video frame
    # by frame
    success, image = vid.read()
    
    if ((temp_frame==num_frames)or(RMSE_val_raw==0)):
            break
    
    temp_frame += 1
    #convert to tensor
    
    try:
        if image is not None:
            frame_tens = convert_tensor(image)
            
            #fix the frame before comparing with raw image
            frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
            
            #Calculate RMSE            
            RMSE_val_raw = sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5
            
            if(RMSE_val_raw<RMSE_val):
                RMSE_val=RMSE_val_raw
                flipped="no"
                frame_nums = temp_frame
                vid_loc = video
    except:
        pass


KeyboardInterrupt: 

In [33]:
RMSE_val_raw

118.19053

In [37]:
(num_frames/100)/60


8.85

In [34]:
temp_frame

100

In [13]:
video = m_st_1_dat["video_location"][im_num]
vidcap = cv2.VideoCapture(video)
temp_frame = 0


    #Read the video frame by frame
    success,image = vidcap.read()
    
    temp_frame =+ 1
    #convert to tensor
    frame_tens = convert_tensor(image)
    
    #fix the frame before comparing with raw image
    frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
    
    #Calculate RMSE            
    RMSE_val_raw = sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5
    
    if(RMSE_val_raw<RMSE_val):
        RMSE_val=RMSE_val_raw
        flipped="no"
        frame_nums = temp_frame
        vid_loc = video

vidcap.release()



TypeError: pic should be PIL Image or ndarray. Got <class 'NoneType'>

In [ ]:
       

im_name = [img_list[im_num]]

sim_dat = pd.DataFrame({'image_name':im_name,'video_location':vid_loc,'flipped':flipped,'frame_number':frame_nums,'RMSE':RMSE_val})

sim_dat.to_csv(score_dir + img_list[im_num].split(".")[0] + "_image_sim_scores.csv",index=False)

In [ ]:
for vid_num in range(num_vals):
    try:
        #read in video
        video = videos[vid_num]
        vidcap = cv2.VideoCapture(video)
        
        for frame_num in range(frame_step,int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT)),frame_step):
            try:
                #extract frame
                frame = frame_num
                vidcap.set(1, frame)
                success,image = vidcap.read()

                #convert to tensor
                frame_tens = convert_tensor(image)

                #fix the frame before comparing with raw image
                frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
                
                #compare similarity

                #root mean square error
                RMSE_val_raw = sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5
                RMSE_val_raw_flipped = sum(sum(sum((frame_tens_fixed - img_tens_flipped)**2))).numpy()**0.5
                
                if(RMSE_val_raw<RMSE_val):
                    RMSE_val=RMSE_val_raw
                    flipped="no"
                    frame_nums = frame_num
                    vid_loc = video
                    
                if(RMSE_val_raw_flipped<RMSE_val):
                    RMSE_val=RMSE_val_raw_flipped
                    flipped="yes"
                    frame_nums = frame_num
                    vid_loc = video
            except:
                pass
    except:
        pass
                    

im_name = [img_list[im_num]]

sim_dat = pd.DataFrame({'image_name':im_name,'video_location':vid_loc,'flipped':flipped,'frame_number':frame_nums,'RMSE':RMSE_val})

sim_dat.to_csv(score_dir + img_list[im_num].split(".")[0] + "_image_sim_scores.csv",index=False)

In [3]:
#code to create vid locations and image names csvs

# #load image list
# img_list = os.listdir(img_dir)
# #save it
# pd.DataFrame({'image_name':img_list}).to_csv("../../data/Kakadu_fish/other/matching_images/image_names.csv",index=False)

# #load video list
# videos_l =  [glob.glob(vid_dir + "*/*/*/*/" + "*.*"),glob.glob(vid_dir + "*/*/*/*/*/" + "*.MP4")]
# videos = [item for sublist in videos_l for item in sublist]

# Fix up video locs
# videos_fixed = list(
#     map(lambda item: item.replace("\\","/").replace("//","/"), videos)
# )

# Get video names 
# video_name = list(
#     map(lambda item: item.split('/')[-1], videos_fixed)
# )

# #get video types
# file_type = list(
#     map(lambda item: item.split('.')[-1], video_name)
# )

#save it
#pd.DataFrame({'video_location':videos_fixed,"video_name":video_name,"file_type":file_type}).to_csv("../../data/Kakadu_fish/other/matching_images/vid_locations.csv",index=False)

In [14]:
#choose vid and image number
vid_num = 0
im_num = 0
#choose frame number/s
frame_num = 0

In [45]:
#read in video
video = videos[vid_num]
vidcap = cv2.VideoCapture(video)

In [104]:
#read in video
video = videos[vid_num]
vidcap = cv2.VideoCapture(video)

#extract frame
frame = frame_num
vidcap.set(1, frame)
success,image = vidcap.read()

#convert to tensor
frame_tens = convert_tensor(image)

#fix the frame before comparing with raw image
frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)

#Load image
img = Image.open(img_dir +img_list[im_num])
img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
img_tens = convert_tensor(img)
img_tens_flipped = convert_tensor(img_flipped)

#prepare images for similarity metrics
frame_sim = skimage.img_as_float(convert_image(frame_tens_fixed))
img_sim = skimage.img_as_float(img)

#compare similarity

#root mean square error
RMSE_s         = sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5
RMSE_s_flipped = sum(sum(sum((frame_tens_fixed - img_tens_flipped)**2))).numpy()**0.5

#ssim
ssim = skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2)
ssim_flipped = skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2)
#ssim_g = skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2,
#                                               gaussian_weights=True, sigma=1.5,use_sample_covariance=False)

In [151]:
#number of vals
num_vals = 10

In [ ]:
#Create loop over videos

#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_vals = []
SSIM_vals = []

#Load image
img = Image.open(img_dir +img_list[im_num])
img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
img_tens = convert_tensor(img)
img_tens_flipped = convert_tensor(img_flipped)


for vid_num in range(num_vals):
    #read in video
    video = videos[vid_num]
    vidcap = cv2.VideoCapture(video)
    
    #extract frame
    frame = frame_num
    vidcap.set(1, frame)
    success,image = vidcap.read()

    #convert to tensor
    frame_tens = convert_tensor(image)

    #fix the frame before comparing with raw image
    frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
    

    #prepare images for similarity metrics
    frame_sim = skimage.img_as_float(convert_image(frame_tens_fixed))
    img_sim = skimage.img_as_float(img)

    #compare similarity

    #root mean square error
    RMSE_vals.append(sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5)
    RMSE_vals.append(sum(sum(sum((frame_tens_fixed - img_tens_flipped)**2))).numpy()**0.5)

    #ssim
    SSIM_vals.append(skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2))
    SSIM_vals.append(skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2))
    
flipped = ["no","yes"]*num_vals
im_name = [img_list[im_num]]*num_vals*2
vid_loc = np.repeat(videos[:num_vals],2)

In [84]:
#Create loop over videos and their frames every 16 seconds
num_vals =2
frame_step=18000

#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_vals = []
frame_nums = []

#Load image
img = Image.open(img_dir +img_list[im_num])
img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
img_tens = convert_tensor(img)
img_tens_flipped = convert_tensor(img_flipped)


for vid_num in range(num_vals):
    #read in video
    video = videos[vid_num]
    vidcap = cv2.VideoCapture(video)
    
    for frame_num in range(frame_step,int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))-frame_step,frame_step):
        #extract frame
        frame = frame_num
        vidcap.set(1, frame)
        success,image = vidcap.read()

        #convert to tensor
        frame_tens = convert_tensor(image)

        #fix the frame before comparing with raw image
        frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
        
        #compare similarity

        #root mean square error
        RMSE_vals.append(torch.sum((frame_tens_fixed- img_tens)**2)**0.5)
        RMSE_vals.append(torch.sum((frame_tens_fixed - img_tens_flipped)**2)**0.5)
        
        #update value
        vid_loc.append(video)
        frame_nums.append(frame_num)

num_ims = int(len(RMSE_vals)/2)
flipped = ["no","yes"]*num_ims
im_name = [img_list[im_num]]*len(RMSE_vals)


In [92]:
#Create loop over videos and their frames every 16 seconds
num_vals =10
im_num=1

#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_val = 1000
frame_nums = []
frame_step=18000

#Load image
img = Image.open(img_dir +img_list[im_num])
img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
img_tens = convert_tensor(img)
img_tens_flipped = convert_tensor(img_flipped)


for vid_num in range(num_vals):
    #read in video
    video = videos[vid_num]
    vidcap = cv2.VideoCapture(video)
    
    for frame_num in range(frame_step,int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))-frame_step,frame_step):
        #extract frame
        frame = frame_num
        vidcap.set(1, frame)
        success,image = vidcap.read()

        #convert to tensor
        frame_tens = convert_tensor(image)

        #fix the frame before comparing with raw image
        frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
        
        #compare similarity

        #root mean square error
        #RMSE_val_raw = torch.sum((frame_tens_fixed- img_tens)**2)**0.5
        RMSE_val_raw = sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5
        #RMSE_val_raw_flipped = torch.sum((frame_tens_fixed - img_tens_flipped)**2)**0.5
        RMSE_val_raw_flipped = sum(sum(sum((frame_tens_fixed - img_tens_flipped)**2))).numpy()**0.5
        
        if(RMSE_val_raw<RMSE_val):
            RMSE_val=RMSE_val_raw
            flipped="no"
            frame_nums = frame_num
            vid_loc = video
            
        if(RMSE_val_raw_flipped<RMSE_val):
            RMSE_val=RMSE_val_raw_flipped
            flipped="yes"
            frame_nums = frame_num
            vid_loc = video
            
        

RMSE_val = RMSE_val
im_name = [img_list[im_num]]

484.02527

In [82]:
int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))

53100

In [78]:
RMSE_val.numpy()

IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [66]:
vid_loc_f = np.repeat(vid_loc,2)
frame_nums_f = np.repeat(frame_nums,2)

In [68]:
frame_nums_f

array([ 1000,  1000,  2000,  2000,  3000,  3000,  4000,  4000,  5000,
        5000,  6000,  6000,  7000,  7000,  8000,  8000,  9000,  9000,
       10000, 10000, 11000, 11000, 12000, 12000, 13000, 13000, 14000,
       14000, 15000, 15000, 16000, 16000, 17000, 17000, 18000, 18000,
       19000, 19000, 20000, 20000, 21000, 21000, 22000, 22000, 23000,
       23000, 24000, 24000, 25000, 25000, 26000, 26000, 27000, 27000,
       28000, 28000, 29000, 29000, 30000, 30000, 31000, 31000, 32000,
       32000, 33000, 33000, 34000, 34000, 35000, 35000, 36000, 36000,
       37000, 37000, 38000, 38000, 39000, 39000, 40000, 40000, 41000,
       41000, 42000, 42000, 43000, 43000, 44000, 44000, 45000, 45000,
       46000, 46000, 47000, 47000, 48000, 48000, 49000, 49000, 50000,
       50000, 51000, 51000, 52000, 52000, 53000, 53000,  1000,  1000,
        2000,  2000,  3000,  3000,  4000,  4000,  5000,  5000,  6000,
        6000,  7000,  7000,  8000,  8000,  9000,  9000, 10000, 10000,
       11000, 11000,

In [40]:
#Create loop over images

#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_vals = []
images = []

num_vals = 100

for im_num in range(num_vals):
    img = Image.open(img_dir +img_list[im_num])
    img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
    img_tens = convert_tensor(img)
    img_tens_flipped = convert_tensor(img_flipped)
    images.append(img_tens)
    images.append(img_tens_flipped)

In [19]:
img_tens_flipped.shape

torch.Size([3, 1080, 1920])

In [41]:
images = torch.zeros(3, 1080,1920,num_vals*2)

for im_num in range(num_vals):
    img = Image.open(img_dir +img_list[im_num])
    img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
    img_tens = convert_tensor(img)
    img_tens_flipped = convert_tensor(img_flipped)
    images[:,:,:,2*im_num] = img_tens
    images[:,:,:,2*im_num+1] = img_tens_flipped

In [43]:
torch.save(images,score_dir+"tensor_test.pt")

In [44]:
tensor = torch.load(score_dir+"tensor_test.pt")

KeyboardInterrupt: 

In [ ]:
#Create loop over images
num_vals = 100
#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_vals = []
images = torch.zeros(3, 1080,1920,num_vals*2)



for im_num in range(num_vals):
    img = Image.open(img_dir +img_list[im_num])
    img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
    img_tens = convert_tensor(img)
    img_tens_flipped = convert_tensor(img_flipped)
    images.append(img_tens)
    images.append(img_tens_flipped)

In [25]:
torch.sum(img_tens_flipped)**0.5

tensor(1680.2798)

In [12]:
len(images)

200

In [34]:
#now loop into a video
for i in range(len(images)):
    RMSE_vals.append(torch.sum((images[i] - img_tens)**2)**0.5)

In [31]:
#now loop into a video
for i in range(len(images)):
    RMSE_vals.append(torch.sum(torch.sub(images[i],img_tens)**2)**0.5)

In [33]:
RMSE_vals = list(
    map(lambda item: torch.sum(torch.sub(item,img_tens)**2)**0.5, images)
)

In [15]:
RMSE_vals

[620.8367,
 749.3677,
 564.99097,
 924.7759,
 642.0692,
 746.4472,
 626.1554,
 885.65094,
 602.82855,
 858.8369,
 423.48825,
 839.6982,
 524.25134,
 931.33374,
 721.40424,
 813.29095,
 557.0763,
 942.39624,
 593.92163,
 960.938,
 635.36346,
 723.48,
 586.3723,
 755.32776,
 494.06342,
 778.1827,
 604.2341,
 832.31354,
 422.28458,
 838.57495,
 568.71893,
 790.15594,
 625.1102,
 874.88336,
 554.7247,
 832.2555,
 506.47095,
 840.862,
 552.11554,
 767.4405,
 586.52893,
 859.7249,
 594.1076,
 688.895,
 608.28375,
 673.88324,
 599.7029,
 954.2038,
 526.25055,
 640.2568,
 621.51715,
 846.4226,
 570.9197,
 799.68115,
 563.3287,
 790.0628,
 515.31396,
 784.5137,
 453.00342,
 794.5029,
 524.9138,
 707.14825,
 406.49442,
 773.62524,
 640.8643,
 752.6155,
 604.89655,
 864.3926,
 575.1621,
 811.1348,
 528.8492,
 670.57697,
 498.20523,
 858.02246,
 594.17395,
 711.6141,
 572.2081,
 670.96356,
 403.0828,
 808.8181,
 607.1766,
 593.4505,
 540.2682,
 940.7033,
 493.6394,
 856.56366,
 580.85504,
 759.900

In [35]:
import json

In [37]:
import pickle

In [38]:
with open("test","wb") as fp:
    pickle.dump(images,fp)
    

In [39]:
with open("test", "rb") as fp:   # Unpickling
   b = pickle.load(fp)

KeyboardInterrupt: 

In [152]:
#Create loop over images

#create vectors
im_name = []
vid_loc = []
flipped = []
RMSE_vals = []


#Load image
img = Image.open(img_dir +img_list[im_num])
img_flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
img_tens = convert_tensor(img)
img_tens_flipped = convert_tensor(img_flipped)


for vid_num in range(num_vals):
    #read in video
    video = videos[vid_num]
    vidcap = cv2.VideoCapture(video)
    
    #extract frame
    frame = frame_num
    vidcap.set(1, frame)
    success,image = vidcap.read()

    #convert to tensor
    frame_tens = convert_tensor(image)

    #fix the frame before comparing with raw image
    frame_tens_fixed = torch.stack((frame_tens[2,:,:],frame_tens[1,:,:],frame_tens[0,:,:]),dim=0)
    

    #prepare images for similarity metrics
    frame_sim = skimage.img_as_float(convert_image(frame_tens_fixed))
    img_sim = skimage.img_as_float(img)

    #compare similarity

    #root mean square error
    RMSE_vals.append(sum(sum(sum((frame_tens_fixed - img_tens)**2))).numpy()**0.5)
    RMSE_vals.append(sum(sum(sum((frame_tens_fixed - img_tens_flipped)**2))).numpy()**0.5)

    #ssim
    SSIM_vals.append(skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2))
    SSIM_vals.append(skimage.metrics.structural_similarity(img_sim,frame_sim,data_range=1,channel_axis=2))
    
flipped = ["no","yes"]*num_vals
im_name = [img_list[im_num]]*num_vals*2
vid_loc = np.repeat(videos[:num_vals],2)

In [153]:
sim_dat = pd.DataFrame({'image_name':im_name,'video_location':vid_loc,'flipped':flipped,'RMSE':RMSE_vals,'SSIM':SSIM_vals})